# Process GPI TDM H+T by City

Computes city-level median household income from ACS block-group data, then uses it
to normalize city H+T (Housing + Transportation) costs into an income-share metric.
The `HPLUST{YYYY}` fields contain the 0-1 income share `(HTCOST{YYYY} × 12) / MEDINC{YYYY}`.
`HTCOST{YYYY}` is the monthly combined H+T cost. `MEDINC{YYYY}` and `TOTPOP{YYYY}` are also exported.

**Workflow**
1. Setup — imports, config, output directories
2. Load Inputs — city boundaries, H+T CSV, derive target years
3. Build 2020 Block Group Reference — fetch via pygris, persist to GDB (skipped if already exists)
4. Fetch ACS Block Group Data by Year — `get_census` per year/county
5. Estimate Missing ACS Years — interpolate internal gaps; extrapolate edges via OLS trend
6. Assign Block Groups to Cities — arcpy centroid spatial join (`HAVE_THEIR_CENTER_IN`)
7. Aggregate to City-Level Income — population-weighted median income
8. Compute H+T Income Share and Build Export — HTCOST (monthly), HPLUST (0-1 share), MEDINC, TOTPOP
9. Validate Export

## 1. Setup

In [1]:
import arcpy
from arcpy import env
import os
import re
import numpy as np
import pandas as pd
from arcgis import GIS
from arcgis.features import GeoAccessor, GeoSeriesAccessor
import geopandas as gpd
from pygris import block_groups
from pygris.data import get_census
from shapely.geometry import MultiPolygon, Polygon

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "90%"
pd.options.display.max_columns = None


In [2]:
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY")

if not CENSUS_API_KEY:
    raise ValueError("CENSUS_API_KEY is not set in the ArcGIS Pro Python environment.")


In [3]:
# Output directories
outputs = [".\\Outputs", "scratch.gdb", "Affordability_Housing_Transportation_Costs.gdb"]

if not os.path.exists(outputs[0]):
    os.makedirs(outputs[0])

gdb = os.path.join(outputs[0], outputs[1])
gdb2 = os.path.join(outputs[0], outputs[2])

if not arcpy.Exists(gdb):
    arcpy.CreateFileGDB_management(outputs[0], outputs[1])

if not arcpy.Exists(gdb2):
    arcpy.CreateFileGDB_management(outputs[0], outputs[2])


In [4]:
# Config for the city-level ACS / block-group / income workflow.
# All new cells in this notebook read from this dictionary.
HT_INCOME_CONFIG = {
    "ht_csv_path": r".\Inputs\H +T Costs for Dashboard 2019-2023 (GPI & TDM) - Composite H + T Metric.csv",
    "bg_reference_gdb": r".\Inputs\block_groups_2020.gdb",
    "bg_reference_name": "bg_2020_ut_5county",
    "bg_projected_name": "bg_2020_projected",
    "bg_city_join_name": "bg_city_spatial_join",
    "state": "UT",
    "state_fips": "49",
    "county_fips": ["003", "011", "035", "049", "057"],
    "county_names": ["BOX ELDER", "DAVIS", "SALT LAKE", "UTAH", "WEBER"],
    "acs_dataset": "acs/acs5",
    "acs_vars": {"median_income": "B19013_001E", "population": "B02001_001E"},
    "year_prefixes": ["HCOST", "TCOST", "HPLUST"],
    "target_years_override": None,
    "pygris_cache": True,
    "share_scale": "0-1",
    "city_key": "CITYAREA",
    "bg_key": "GEOID",
    "workshop_area_lookup": {
        "Box Elder (WFRC)": "Box Elder Wfrc",
        "North Davis County": "Davis County North",
        "South Davis County": "Davis County South",
        "North Salt Lake County": "Salt Lake County North",
        "Southwest Salt Lake County": "Salt Lake County Sw",
        "Southeast Salt Lake County": "Salt Lake County Se",
        "North Weber County": "Weber County North",
        "South Weber County": "Weber County South",
        "Central Utah County": "Utah County Central",
        "North Utah County": "Utah County North",
        "South Utah County": "Utah County South",
    },
}


## 2. Load Inputs

In [5]:
# City boundary SEDF — used for export geometry and the spatial join.
# CITY_NAME is added as CITYAREA immediately so the join key is consistent throughout.
city_area_shp = pd.DataFrame.spatial.from_featureclass(
    r".\Inputs\city_area_with_workshop_areas.shp"
)

if (
    HT_INCOME_CONFIG["city_key"] not in city_area_shp.columns
    and "CITY_NAME" in city_area_shp.columns
):
    city_area_shp[HT_INCOME_CONFIG["city_key"]] = city_area_shp["CITY_NAME"]

print("City area rows:", len(city_area_shp))
city_area_shp.head()


City area rows: 109


,FID,CITY_NAME,SUBAREA,CO_NAME,SHAPE,CITYAREA
0,0,Alpine,North Utah County,UTAH,"{""rings"": [[[433081.68319999985, 4477344.8058]...",Alpine
1,1,Alta,NA,SALT LAKE,"{""rings"": [[[449359.0999999996, 4492074], [449...",Alta
2,2,American Fork,North Utah County,UTAH,"{""rings"": [[[433815.22979999986, 4466632.21309...",American Fork
3,3,Balance of BOX ELDER,NA,BOX ELDER,"{""rings"": [[[417479, 4578625.4], [417425.5, 45...",Balance of BOX ELDER
4,4,Benjamin,South Utah County,UTAH,"{""rings"": [[[438601.23319999967, 4437471.4559]...",Benjamin


In [6]:
# H+T cost CSV. Replace -1 sentinel with 0 (no data for that city-year).
# 'City Area' is added as CITYAREA here; 'County' is dropped.
# Heads up: columns read in as object dtype — replace() handles the -1 sentinel before any numeric ops.
ht_df = pd.read_csv(HT_INCOME_CONFIG["ht_csv_path"])
ht_df = ht_df.replace(-1, 0)

if HT_INCOME_CONFIG["city_key"] not in ht_df.columns and "City Area" in ht_df.columns:
    ht_df[HT_INCOME_CONFIG["city_key"]] = ht_df["City Area"]

if "City Area" in ht_df.columns:
    ht_df = ht_df.drop(columns=["City Area"])

if "County" in ht_df.columns:
    ht_df = ht_df.drop(columns=["County"])

print("H+T rows:", len(ht_df))
ht_df.head()


H+T rows: 101


,HCOST2019,HCOST2020,HCOST2021,HCOST2022,HCOST2023,HCOST2024,TCOST2019,TCOST2020,TCOST2021,TCOST2022,TCOST2023,TCOST2024,HPLUST2019,HPLUST2020,HPLUST2021,HPLUST2022,HPLUST2023,HPLUST2024,CITYAREA
0,1829.0,1858.0,2074.0,2640.0,2867.0,4341.0,762.0,608.0,701.0,716.0,738.0,749.0,2592.0,2466.0,2776.0,3356.0,3604.0,5090.0,Alpine
1,0.0,0.0,0.0,0.0,0.0,0.0,504.0,336.0,408.0,447.0,474.0,441.0,0.0,0.0,0.0,0.0,0.0,0.0,Alta
2,1521.0,1600.0,1788.0,2213.0,2400.0,3366.0,429.0,352.0,394.0,406.0,418.0,426.0,1950.0,1951.0,2182.0,2619.0,2819.0,3792.0,American Fork
3,0.0,0.0,0.0,0.0,0.0,0.0,1235.0,914.0,1014.0,1140.0,1340.0,1141.0,0.0,0.0,0.0,0.0,0.0,0.0,Balance of BOX ELDER
4,0.0,0.0,0.0,0.0,0.0,0.0,587.0,486.0,550.0,551.0,558.0,536.0,0.0,0.0,0.0,0.0,0.0,0.0,Benjamin


In [7]:
# Derive target years from HCOST / TCOST / HPLUST column suffixes.
target_years = sorted(
    {
        int(col[-4:])
        for col in ht_df.columns
        if col[-4:].isdigit()
        and any(col.startswith(prefix) for prefix in HT_INCOME_CONFIG["year_prefixes"])
    }
)

if HT_INCOME_CONFIG["target_years_override"] is not None:
    target_years = HT_INCOME_CONFIG["target_years_override"]

print("Detected H+T years:", target_years)
print("Using override    :", HT_INCOME_CONFIG["target_years_override"] is not None)


Detected H+T years: [2019, 2020, 2021, 2022, 2023, 2024]
Using override    : False


In [8]:
# Validation: confirm city names in CSV match city names in shapefile.
# Cities in the shapefile but not the CSV are geometry-only rows (no H+T data) — expected.
# Cities in the CSV but not the shapefile would be a spelling mismatch requiring investigation.
ht_cities = set(ht_df[HT_INCOME_CONFIG["city_key"]].dropna())
shp_col = (
    HT_INCOME_CONFIG["city_key"]
    if HT_INCOME_CONFIG["city_key"] in city_area_shp.columns
    else "CITY_NAME"
)
shape_cities = set(city_area_shp[shp_col].dropna())

in_csv_not_shp = sorted(ht_cities - shape_cities)
in_shp_not_csv = sorted(shape_cities - ht_cities)

print("In CSV but NOT shapefile (investigate if non-empty):", in_csv_not_shp)
print("In shapefile but NOT CSV (geometry-only rows, expected):", in_shp_not_csv)


In CSV but NOT shapefile (investigate if non-empty): []
In shapefile but NOT CSV (geometry-only rows, expected): ['Camp Williams', 'Davis County', 'Lake Mountain', 'SL County East Cyns', 'South Cedar Valley', 'South Goshen Valley', 'Utah Lake', 'West Mountain']


## 3. Build 2020 Block Group Reference

A single 2020 Census block group layer is used as the stable spatial reference for all
target years. It is fetched once via `pygris`, normalised to `MultiPolygon` (required by
the OpenFileGDB Fiona driver), and persisted to a file GDB. Subsequent runs skip the
fetch if the layer already exists.

In [9]:
bg_reference_fc = os.path.join(
    HT_INCOME_CONFIG["bg_reference_gdb"], HT_INCOME_CONFIG["bg_reference_name"]
)

if not arcpy.Exists(HT_INCOME_CONFIG["bg_reference_gdb"]):
    arcpy.CreateFileGDB_management(r".\Inputs", "block_groups_2020.gdb")

if not arcpy.Exists(bg_reference_fc):
    bg_2020_gdf = block_groups(
        state=HT_INCOME_CONFIG["state"], year=2020, cache=HT_INCOME_CONFIG["pygris_cache"]
    )
    bg_2020_gdf = bg_2020_gdf[bg_2020_gdf["COUNTYFP"].isin(HT_INCOME_CONFIG["county_fips"])].copy()
    bg_2020_gdf = bg_2020_gdf[
        ["GEOID", "STATEFP", "COUNTYFP", "TRACTCE", "BLKGRPCE", "geometry"]
    ].copy()

    # Normalise mixed Polygon/MultiPolygon so the OpenFileGDB driver has a consistent type.
    bg_2020_gdf["geometry"] = bg_2020_gdf["geometry"].apply(
        lambda g: MultiPolygon([g]) if isinstance(g, Polygon) else g
    )
    bg_2020_gdf.to_file(
        HT_INCOME_CONFIG["bg_reference_gdb"],
        layer=HT_INCOME_CONFIG["bg_reference_name"],
        driver="OpenFileGDB",
    )
    print("Exported:", bg_reference_fc)
    print("Rows    :", len(bg_2020_gdf))
    print("Counties:", sorted(bg_2020_gdf["COUNTYFP"].unique().tolist()))
    print("CRS     :", bg_2020_gdf.crs)
else:
    print("Using existing block group reference:", bg_reference_fc)


Using existing block group reference: .\Inputs\block_groups_2020.gdb\bg_2020_ut_5county


In [10]:
# Read back as arcgis SEDF. Cast GEOID to str here — all downstream joins depend on this.
block_groups_ref = pd.DataFrame.spatial.from_featureclass(bg_reference_fc)
block_groups_ref[HT_INCOME_CONFIG["bg_key"]] = block_groups_ref[HT_INCOME_CONFIG["bg_key"]].astype(
    str
)

print("Columns:", block_groups_ref.columns.tolist())
print("Shape  :", block_groups_ref.shape)
block_groups_ref.head()


Columns: ['OBJECTID', 'GEOID', 'STATEFP', 'COUNTYFP', 'TRACTCE', 'BLKGRPCE', 'SHAPE']
Shape  : (1547, 7)


,OBJECTID,GEOID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,SHAPE
0,1,490351113061,49,035,111306,1,"{""rings"": [[[-111.83382099999994, 40.615462000..."
1,2,490351113062,49,035,111306,2,"{""rings"": [[[-111.82158699999997, 40.607759000..."
2,3,490351114001,49,035,111400,1,"{""rings"": [[[-111.88258799999994, 40.718417000..."
3,4,490351114002,49,035,111400,2,"{""rings"": [[[-111.88262299999997, 40.712825000..."
4,5,490351114003,49,035,111400,3,"{""rings"": [[[-111.88267499999995, 40.704964000..."


## 4. Fetch ACS Block Group Data by Year

For each target year, `get_census` retrieves median household income (`B19013_001E`)
and total population (`B02001_001E`) at the block-group level across all 5 counties.
ACS data is left-joined onto the 2020 BG reference geometry.

Years where the API call fails are logged but do not stop execution — missing years are
handled in the next section.

In [11]:
bg_acs_by_year = {}
acs_year_status = []

for year in target_years:
    county_frames = []
    year_ok = True
    year_message = "ok"

    for county_fips in HT_INCOME_CONFIG["county_fips"]:
        try:
            county_df = get_census(
                dataset=HT_INCOME_CONFIG["acs_dataset"],
                variables=list(HT_INCOME_CONFIG["acs_vars"].values()),
                year=year,
                params={
                    "for": "block group:*",
                    "in": f"state:{HT_INCOME_CONFIG['state_fips']} county:{county_fips}",
                    "key": CENSUS_API_KEY,
                },
                return_geoid=True,
                guess_dtypes=True,
            )
            county_frames.append(county_df)
        except Exception as exc:
            year_ok = False
            year_message = str(exc)
            break

    if year_ok and county_frames:
        acs_df = pd.concat(county_frames, ignore_index=True)
        acs_df = acs_df.rename(
            columns={
                HT_INCOME_CONFIG["acs_vars"]["median_income"]: "median_income",
                HT_INCOME_CONFIG["acs_vars"]["population"]: "population",
            }
        )
        acs_df[HT_INCOME_CONFIG["bg_key"]] = acs_df[HT_INCOME_CONFIG["bg_key"]].astype(str)
        acs_df["median_income"] = pd.to_numeric(acs_df["median_income"], errors="coerce")
        acs_df["population"] = pd.to_numeric(acs_df["population"], errors="coerce")
        acs_df.loc[acs_df["median_income"] <= 0, "median_income"] = np.nan
        acs_df.loc[acs_df["population"] <= 0, "population"] = np.nan
        acs_df["acs_year"] = year
        acs_df["income_source"] = "acs"
        acs_df["population_source"] = "acs"
        acs_df = acs_df[
            [
                HT_INCOME_CONFIG["bg_key"],
                "median_income",
                "population",
                "acs_year",
                "income_source",
                "population_source",
            ]
        ].copy()

        bg_acs_by_year[year] = block_groups_ref.merge(
            acs_df, on=HT_INCOME_CONFIG["bg_key"], how="left"
        )
        acs_year_status.append(
            {
                "year": year,
                "status": "fetched",
                "acs_rows": len(acs_df),
                "matched_bg_rows": bg_acs_by_year[year]["median_income"].notna().sum(),
                "message": "ok",
            }
        )
    else:
        bg_acs_by_year[year] = block_groups_ref.copy()
        bg_acs_by_year[year]["median_income"] = np.nan
        bg_acs_by_year[year]["population"] = np.nan
        bg_acs_by_year[year]["acs_year"] = year
        bg_acs_by_year[year]["income_source"] = np.nan
        bg_acs_by_year[year]["population_source"] = np.nan
        acs_year_status.append(
            {
                "year": year,
                "status": "missing",
                "acs_rows": 0,
                "matched_bg_rows": 0,
                "message": year_message,
            }
        )


In [12]:
# Fetch summary — check for any missing years before proceeding.
acs_year_status_df = pd.DataFrame(acs_year_status)
acs_year_status_df


,year,status,acs_rows,matched_bg_rows,message
0,2019,fetched,1297,1053,ok
1,2020,fetched,1547,1497,ok
2,2021,fetched,1547,1503,ok
3,2022,fetched,1547,1496,ok
4,2023,fetched,1547,1496,ok
5,2024,fetched,1547,1501,ok


In [13]:
# Spot check: confirm ACS fields joined correctly for the first year.
sample_year = target_years[0]
print("Sample year:", sample_year)
print("Shape      :", bg_acs_by_year[sample_year].shape)
bg_acs_by_year[sample_year][
    [HT_INCOME_CONFIG["bg_key"], "median_income", "population", "acs_year", "income_source"]
].head()


Sample year: 2019
Shape      : (1547, 12)


,GEOID,median_income,population,acs_year,income_source
0,490351113061,75438.0,1792.0,2019.0,acs
1,490351113062,128194.0,839.0,2019.0,acs
2,490351114001,68387.0,1330.0,2019.0,acs
3,490351114002,78548.0,1086.0,2019.0,acs
4,490351114003,51789.0,1734.0,2019.0,acs


## 5. Estimate Missing ACS Years

Each block group is processed independently across all target years:
- **Internal gaps** (e.g. 2020 missing when 2019 and 2021 exist) are filled by linear
  interpolation between the two nearest flanking observed values.
- **Edge gaps** (leading or trailing years) are filled by linear extrapolation using an
  OLS trend fitted to **all** observed years for that block group.
- Block groups with fewer than 2 observed years are left as `NaN`.

Observed ACS values are never overwritten. Provenance is tracked via `income_source`
and `population_source`: `"acs"` | `"interpolated"` | `"extrapolated"`.

In [14]:
# Build a long panel (one row per GEOID x year) to drive the fill logic.
acs_panel = []
for year in target_years:
    year_df = bg_acs_by_year[year][
        [
            HT_INCOME_CONFIG["bg_key"],
            "median_income",
            "population",
            "income_source",
            "population_source",
        ]
    ].copy()
    year_df["year"] = year
    acs_panel.append(year_df)

acs_panel_df = pd.concat(acs_panel, ignore_index=True)
print(
    f"Panel shape: {acs_panel_df.shape}  ({len(target_years)} years x {len(block_groups_ref)} BGs)"
)
acs_panel_df.head()


Panel shape: (9282, 6)  (6 years x 1547 BGs)


,GEOID,median_income,population,income_source,population_source,year
0,490351113061,75438.0,1792.0,acs,acs,2019
1,490351113062,128194.0,839.0,acs,acs,2019
2,490351114001,68387.0,1330.0,acs,acs,2019
3,490351114002,78548.0,1086.0,acs,acs,2019
4,490351114003,51789.0,1734.0,acs,acs,2019


In [15]:
filled_panel_parts = []

for geoid, group in acs_panel_df.groupby(HT_INCOME_CONFIG["bg_key"]):
    group = group.sort_values("year").copy()

    for value_col, source_col in [
        ("median_income", "income_source"),
        ("population", "population_source"),
    ]:
        observed = group[["year", value_col]].dropna()

        if len(observed) >= 2:
            years_obs = observed["year"].to_numpy(dtype=float)
            vals_obs = observed[value_col].to_numpy(dtype=float)
            first_year = int(years_obs[0])
            last_year = int(years_obs[-1])

            # 1. Internal gaps — linear interpolation between flanking observed values.
            group[value_col] = group[value_col].interpolate(method="linear", limit_area="inside")
            missing_mask = group[value_col].isna()

            # OLS trend over ALL observed years — used for edge extrapolation only.
            trend_slope, trend_intercept = np.polyfit(years_obs, vals_obs, 1)

            # 2. Left-edge extrapolation.
            left_mask = missing_mask & (group["year"] < first_year)
            if left_mask.any():
                group.loc[left_mask, value_col] = (
                    trend_slope * group.loc[left_mask, "year"] + trend_intercept
                )

            missing_mask = group[value_col].isna()

            # 3. Right-edge extrapolation.
            right_mask = missing_mask & (group["year"] > last_year)
            if right_mask.any():
                group.loc[right_mask, value_col] = (
                    trend_slope * group.loc[right_mask, "year"] + trend_intercept
                )

            # 4. Tag source only for rows that were originally missing and now have a value.
            original_missing = (
                acs_panel_df[
                    (acs_panel_df[HT_INCOME_CONFIG["bg_key"]] == geoid)
                    & (acs_panel_df["year"].isin(group["year"]))
                ][value_col]
                .reset_index(drop=True)
                .isna()
            )
            group = group.reset_index(drop=True)
            for idx in group.index:
                if original_missing.iloc[idx] and pd.notna(group.loc[idx, value_col]):
                    if first_year < group.loc[idx, "year"] < last_year:
                        group.loc[idx, source_col] = "interpolated"
                    else:
                        group.loc[idx, source_col] = "extrapolated"

        # Rounding and sign guards (applied regardless of whether fill ran).
        if value_col == "median_income":
            group[value_col] = group[value_col].round(0)
            group.loc[group[value_col] <= 0, value_col] = np.nan

        if value_col == "population":
            group[value_col] = group[value_col].round(0)
            group.loc[group[value_col] <= 0, value_col] = np.nan

    filled_panel_parts.append(group)

acs_panel_filled_df = pd.concat(filled_panel_parts, ignore_index=True)
acs_panel_filled_df.head()


,GEOID,median_income,population,income_source,population_source,year
0,490039601001,65682.0,766.0,acs,acs,2019
1,490039601001,69875.0,874.0,acs,acs,2020
2,490039601001,82500.0,1348.0,acs,acs,2021
3,490039601001,88542.0,1174.0,acs,acs,2022
4,490039601001,92262.0,1150.0,acs,acs,2023


In [16]:
# Fill provenance summary by year.
fill_summary_df = (
    acs_panel_filled_df.groupby("year")
    .agg(
        income_acs=pd.NamedAgg(column="income_source", aggfunc=lambda s: (s == "acs").sum()),
        income_interpolated=pd.NamedAgg(
            column="income_source", aggfunc=lambda s: (s == "interpolated").sum()
        ),
        income_extrapolated=pd.NamedAgg(
            column="income_source", aggfunc=lambda s: (s == "extrapolated").sum()
        ),
        pop_acs=pd.NamedAgg(column="population_source", aggfunc=lambda s: (s == "acs").sum()),
        pop_interpolated=pd.NamedAgg(
            column="population_source", aggfunc=lambda s: (s == "interpolated").sum()
        ),
        pop_extrapolated=pd.NamedAgg(
            column="population_source", aggfunc=lambda s: (s == "extrapolated").sum()
        ),
    )
    .reset_index()
)
fill_summary_df


,year,income_acs,income_interpolated,income_extrapolated,pop_acs,pop_interpolated,pop_extrapolated
0,2019,1058,0,474,1063,0,474
1,2020,1517,11,19,1543,0,4
2,2021,1523,19,5,1545,0,2
3,2022,1517,25,5,1547,0,0
4,2023,1517,20,10,1547,0,0
5,2024,1519,0,28,1547,0,0


In [17]:
# Spot check: all years for one GEOID should have sensible values and sources.
sample_geoid = acs_panel_filled_df[HT_INCOME_CONFIG["bg_key"]].iloc[0]
acs_panel_filled_df[acs_panel_filled_df[HT_INCOME_CONFIG["bg_key"]] == sample_geoid].sort_values(
    "year"
)


,GEOID,median_income,population,income_source,population_source,year
0,490039601001,65682.0,766.0,acs,acs,2019
1,490039601001,69875.0,874.0,acs,acs,2020
2,490039601001,82500.0,1348.0,acs,acs,2021
3,490039601001,88542.0,1174.0,acs,acs,2022
4,490039601001,92262.0,1150.0,acs,acs,2023
5,490039601001,101100.0,1173.0,acs,acs,2024


In [18]:
# Write filled values back into bg_acs_by_year, replacing the raw ACS columns.
for year in target_years:
    year_fill = acs_panel_filled_df[acs_panel_filled_df["year"] == year][
        [
            HT_INCOME_CONFIG["bg_key"],
            "median_income",
            "population",
            "income_source",
            "population_source",
        ]
    ].copy()

    base_cols = [
        col
        for col in bg_acs_by_year[year].columns
        if col not in ["median_income", "population", "income_source", "population_source"]
    ]
    bg_acs_by_year[year] = bg_acs_by_year[year][base_cols].merge(
        year_fill, on=HT_INCOME_CONFIG["bg_key"], how="left"
    )


## 6. Assign Block Groups to Cities (Spatial Join)

Block groups are assigned to exactly one city using a centroid-based spatial join
(`HAVE_THEIR_CENTER_IN`). This avoids double-counting from polygon-intersection joins.
Unmatched block groups are logged and excluded from aggregation.

The BG reference is in GCS NAD83 (EPSG:4269); the city shapefile is in UTM Zone 12N
(EPSG:26912). A `CopyFeatures` + `Project` pattern re-projects the BGs — `CopyFeatures`
is required first to avoid ERROR 001489 (topology participation prevents direct `Project`).

In [19]:
city_fc = r".\Inputs\city_area_with_workshop_areas.shp"
bg_copy_fc = os.path.join(gdb, HT_INCOME_CONFIG["bg_projected_name"] + "_copy")
bg_projected_fc = os.path.join(gdb, HT_INCOME_CONFIG["bg_projected_name"])
bg_city_join_fc = os.path.join(gdb, HT_INCOME_CONFIG["bg_city_join_name"])
bg_join_input_fc = bg_reference_fc

bg_sr = arcpy.Describe(bg_reference_fc).spatialReference
city_sr = arcpy.Describe(city_fc).spatialReference

print("BG SR  :", bg_sr.name, bg_sr.factoryCode)
print("City SR:", city_sr.name, city_sr.factoryCode)

if bg_sr.factoryCode != city_sr.factoryCode:
    if arcpy.Exists(bg_copy_fc):
        arcpy.management.Delete(bg_copy_fc)
    arcpy.management.CopyFeatures(bg_reference_fc, bg_copy_fc)

    if arcpy.Exists(bg_projected_fc):
        arcpy.management.Delete(bg_projected_fc)
    arcpy.management.Project(
        in_dataset=bg_copy_fc, out_dataset=bg_projected_fc, out_coor_system=city_sr
    )
    arcpy.management.Delete(bg_copy_fc)

    bg_join_input_fc = bg_projected_fc
    print("Projected to:", city_sr.name)
else:
    print("CRS match — no projection needed")


BG SR  : GCS_North_American_1983 4269
City SR: NAD_1983_UTM_Zone_12N 26912
Projected to: NAD_1983_UTM_Zone_12N


In [20]:
if arcpy.Exists(bg_city_join_fc):
    arcpy.management.Delete(bg_city_join_fc)

arcpy.analysis.SpatialJoin(
    target_features=bg_join_input_fc,
    join_features=city_fc,
    out_feature_class=bg_city_join_fc,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="HAVE_THEIR_CENTER_IN",
)


<Result '.\\Outputs\\scratch.gdb\\bg_city_spatial_join'>

In [21]:
# Read back the join result; apply workshop-area renames to SUBAREA.
bg_city_lookup = pd.DataFrame.spatial.from_featureclass(bg_city_join_fc)[
    ["GEOID", "CITY_NAME", "SUBAREA", "CO_NAME"]
].copy()
bg_city_lookup.rename(columns={"CITY_NAME": HT_INCOME_CONFIG["city_key"]}, inplace=True)
bg_city_lookup[HT_INCOME_CONFIG["bg_key"]] = bg_city_lookup[HT_INCOME_CONFIG["bg_key"]].astype(str)
bg_city_lookup["CO_NAME"] = bg_city_lookup["CO_NAME"].str.upper()
bg_city_lookup["SUBAREA"] = bg_city_lookup["SUBAREA"].replace(
    HT_INCOME_CONFIG["workshop_area_lookup"]
)

bg_city_lookup.head()


,GEOID,CITYAREA,SUBAREA,CO_NAME
0,490351113061,Cottonwood Heights,Salt Lake County Se,SALT LAKE
1,490351113062,Cottonwood Heights,Salt Lake County Se,SALT LAKE
2,490351114001,South Salt Lake,Salt Lake County North,SALT LAKE
3,490351114002,South Salt Lake,Salt Lake County North,SALT LAKE
4,490351114003,South Salt Lake,Salt Lake County North,SALT LAKE


In [22]:
# Validate: duplicate GEOIDs indicate a join error and must be zero.
dup_count = bg_city_lookup[HT_INCOME_CONFIG["bg_key"]].duplicated().sum()
if dup_count > 0:
    raise ValueError(
        f"Duplicate GEOIDs in bg_city_lookup: {dup_count}. Investigate before continuing."
    )

print("Total BG rows    :", len(bg_city_lookup))
print("Duplicate GEOIDs :", dup_count)
print("Matched BGs      :", bg_city_lookup[HT_INCOME_CONFIG["city_key"]].notna().sum())
print(
    "Unmatched BGs    :",
    bg_city_lookup[HT_INCOME_CONFIG["city_key"]].isna().sum(),
    "(excluded from aggregation)",
)

bg_city_lookup[
    [HT_INCOME_CONFIG["bg_key"], HT_INCOME_CONFIG["city_key"], "SUBAREA", "CO_NAME"]
].head(10)


Total BG rows    : 1547
Duplicate GEOIDs : 0
Matched BGs      : 1521
Unmatched BGs    : 26 (excluded from aggregation)


,GEOID,CITYAREA,SUBAREA,CO_NAME
0,490351113061,Cottonwood Heights,Salt Lake County Se,SALT LAKE
1,490351113062,Cottonwood Heights,Salt Lake County Se,SALT LAKE
2,490351114001,South Salt Lake,Salt Lake County North,SALT LAKE
3,490351114002,South Salt Lake,Salt Lake County North,SALT LAKE
4,490351114003,South Salt Lake,Salt Lake County North,SALT LAKE
5,490351114005,South Salt Lake,Salt Lake County North,SALT LAKE
6,490351114006,South Salt Lake,Salt Lake County North,SALT LAKE
7,490572002022,Ogden,Weber County South,WEBER
8,490351117023,South Salt Lake,Salt Lake County North,SALT LAKE
9,490351117024,South Salt Lake,Salt Lake County North,SALT LAKE


## 7. Aggregate to City-Level Income

For each year, block-group income is aggregated to the city level using a
population-weighted mean:

```
median_income_YYYY = sum(median_income * population) / sum(population)
```

Block groups with missing income, missing population, or zero population are excluded.

In [23]:
city_income_parts = []

for year in target_years:
    year_df = bg_acs_by_year[year].copy()
    year_df[HT_INCOME_CONFIG["bg_key"]] = year_df[HT_INCOME_CONFIG["bg_key"]].astype(str)
    year_df = year_df.merge(bg_city_lookup, on=HT_INCOME_CONFIG["bg_key"], how="left")

    # Keep only rows that can contribute to the weighted average.
    year_df = year_df[
        year_df[HT_INCOME_CONFIG["city_key"]].notna()
        & year_df["median_income"].notna()
        & year_df["population"].notna()
        & (year_df["population"] > 0)
    ].copy()

    year_df["weighted_income"] = year_df["median_income"] * year_df["population"]

    city_year = (
        year_df.groupby(HT_INCOME_CONFIG["city_key"], dropna=False)
        .agg(
            **{
                f"TOTPOP{year}": ("population", "sum"),
                f"MEDINC{year}": ("weighted_income", "sum"),
                f"bg_count_{year}": (HT_INCOME_CONFIG["bg_key"], "count"),
            }
        )
        .reset_index()
    )
    city_year[f"MEDINC{year}"] = (city_year[f"MEDINC{year}"] / city_year[f"TOTPOP{year}"]).round(0)

    city_income_parts.append(city_year)


In [24]:
# Merge all year slices into one wide DataFrame keyed on CITYAREA.
city_income_df = city_income_parts[0].copy()
for part in city_income_parts[1:]:
    city_income_df = city_income_df.merge(part, on=HT_INCOME_CONFIG["city_key"], how="outer")

print("Cities with any income data:", len(city_income_df))
print("Duplicate city rows:", city_income_df[HT_INCOME_CONFIG["city_key"]].duplicated().sum())
city_income_df.head(10)


Cities with any income data: 90
Duplicate city rows: 0


,CITYAREA,TOTPOP2019,MEDINC2019,bg_count_2019,TOTPOP2020,MEDINC2020,bg_count_2020,TOTPOP2021,MEDINC2021,bg_count_2021,TOTPOP2022,MEDINC2022,bg_count_2022,TOTPOP2023,MEDINC2023,bg_count_2023,TOTPOP2024,MEDINC2024,bg_count_2024
0,Alpine,9794.0,135124.0,7.0,10208.0,143417.0,7,9756.0,147254.0,7,9484.0,168423.0,7,9649.0,159966.0,7,9566.0,162409.0,7
1,American Fork,33624.0,84222.0,22.0,33780.0,86650.0,22,34725.0,91186.0,22,36227.0,100624.0,22,37988.0,107877.0,22,39785.0,111523.0,22
2,Balance of BOX ELDER,5295.0,79931.0,4.0,5126.0,85565.0,4,5328.0,92167.0,4,5582.0,103818.0,4,6109.0,108261.0,4,6703.0,116198.0,4
3,Bluffdale,14286.0,99622.0,5.0,14802.0,111512.0,5,16576.0,107601.0,5,17460.0,113575.0,5,18168.0,126394.0,5,18797.0,138150.0,5
4,Bountiful,43322.0,88632.0,31.0,41527.0,89462.0,31,43069.0,96733.0,31,42676.0,103396.0,31,42451.0,105908.0,31,42334.0,101885.0,31
5,Box Elder County North,1704.0,69722.0,1.0,1946.0,72083.0,1,1769.0,76346.0,1,1735.0,113611.0,1,1753.0,88250.0,1,1837.0,94554.0,1
6,Brigham City,15712.0,50706.0,13.0,15886.0,54135.0,13,15897.0,57367.0,13,16264.0,60549.0,13,16465.0,66293.0,13,16625.0,71591.0,13
7,Cedar Fort,NaN,NaN,NaN,386.0,61898.0,1,582.0,102632.0,1,959.0,103250.0,1,1184.0,124286.0,1,2466.0,119643.0,1
8,Cedar Hills,5932.0,95009.0,4.0,6597.0,111954.0,4,6284.0,117943.0,4,6292.0,128312.0,4,6422.0,138708.0,4,6589.0,136171.0,4
9,Centerville,17370.0,93704.0,7.0,16420.0,95754.0,7,16233.0,104084.0,7,16316.0,110059.0,7,16086.0,117270.0,7,16120.0,120122.0,7


In [25]:
# Aggregation spot check: manually verify weighted income for one city-year.
check_year = target_years[0]
check_city = city_income_df[HT_INCOME_CONFIG["city_key"]].dropna().iloc[0]

check_df = bg_acs_by_year[check_year].copy()
check_df[HT_INCOME_CONFIG["bg_key"]] = check_df[HT_INCOME_CONFIG["bg_key"]].astype(str)
check_df = check_df.merge(bg_city_lookup, on=HT_INCOME_CONFIG["bg_key"], how="left")
check_df = check_df[
    (check_df[HT_INCOME_CONFIG["city_key"]] == check_city)
    & check_df["median_income"].notna()
    & check_df["population"].notna()
    & (check_df["population"] > 0)
].copy()
check_df["weighted_income"] = check_df["median_income"] * check_df["population"]

computed = round(check_df["weighted_income"].sum() / check_df["population"].sum(), 0)
stored = city_income_df.loc[
    city_income_df[HT_INCOME_CONFIG["city_key"]] == check_city, f"MEDINC{check_year}"
].iloc[0]

print(f"City: {check_city} | Year: {check_year}")
print(f"Manually computed weighted income: {computed}")
print(f"Value in city_income_df          : {stored}")
print(f"Match: {computed == stored}")


City: Alpine | Year: 2019
Manually computed weighted income: 135124.0
Value in city_income_df          : 135124.0
Match: True


## 8. Compute H+T Income Share and Build Export

Three new fields are computed for each city-year:

```
HTCOSTYYYY      = HCOSTYYYY + TCOSTYYYY          (monthly combined cost, dollars)
HPLUSTYYYY      = (HTCOSTYYYY x 12) / median_income_YYYY   (annual share, 0-1 decimal)
```

`HTCOST{YYYY}` is the monthly sum of housing and transport costs.  
`HPLUST{YYYY}` replaces the original monthly dollar column with the normalized income share,
keeping the output schema backward-compatible with downstream consumers.
All other intermediate derived columns are dropped before export.

Share is set to `NaN` where median income is missing or zero. A guard drops previously
derived columns before merging so this section is safe to re-run.

In [26]:
# Drop previously derived columns so re-runs do not accumulate duplicates.
derived_prefixes = ("MEDINC", "TOTPOP", "bg_count_", "HTCOST")
existing_derived_cols = [col for col in ht_df.columns if col.startswith(derived_prefixes)]
if existing_derived_cols:
    ht_df = ht_df.drop(columns=existing_derived_cols)

# Merge city-level ACS income into the H+T table.
ht_df = ht_df.merge(city_income_df, on=HT_INCOME_CONFIG["city_key"], how="left")

# Compute monthly combined cost (HTCOST) and income share for each year.
# HTCOST{YYYY} = HCOST{YYYY} + TCOST{YYYY}    (monthly dollars)
# HPLUST{YYYY} = (HTCOST{YYYY} * 12) / MEDINC{YYYY}  (0-1 annual share)
for year in target_years:
    htcost_col = f"HTCOST{year}"
    income_col = f"MEDINC{year}"
    ht_df[htcost_col] = np.where(
        ht_df[f"HCOST{year}"].notna() & ht_df[f"TCOST{year}"].notna(),
        ht_df[f"HCOST{year}"] + ht_df[f"TCOST{year}"],
        np.nan,
    )
    ht_df[f"HPLUST{year}"] = np.where(
        ht_df[income_col].notna() & (ht_df[income_col] > 0),
        (ht_df[htcost_col] * 12) / ht_df[income_col],
        np.nan,
    )

# Share range summary.
share_cols = [f"HPLUST{year}" for year in target_years]
htcost_cols = [f"HTCOST{year}" for year in target_years]
income_cols = [f"MEDINC{year}" for year in target_years]

print(
    ht_df[
        [HT_INCOME_CONFIG["city_key"]] + income_cols[:2] + htcost_cols[:2] + share_cols[:2]
    ].head()
)
print()
for year in target_years:
    s = ht_df[f"HPLUST{year}"]
    print(f"{year}  non-null: {s.notna().sum()}  min: {s.min():.4f}  max: {s.max():.4f}")


               CITYAREA  MEDINC2019  MEDINC2020  HTCOST2019  HTCOST2020  \
0                Alpine    135124.0    143417.0      2591.0      2466.0   
1                  Alta         NaN         NaN       504.0       336.0   
2         American Fork     84222.0     86650.0      1950.0      1952.0   
3  Balance of BOX ELDER     79931.0     85565.0      1235.0       914.0   
4              Benjamin         NaN         NaN       587.0       486.0   

   HPLUST2019  HPLUST2020  
0    0.230100    0.206335  
1         NaN         NaN  
2    0.277837    0.270329  
3    0.185410    0.128183  
4         NaN         NaN  

2019  non-null: 83  min: 0.0228  max: 0.4289
2020  non-null: 84  min: 0.0189  max: 0.4239
2021  non-null: 84  min: 0.0232  max: 0.4206
2022  non-null: 84  min: 0.0373  max: 0.4198
2023  non-null: 84  min: 0.0263  max: 0.4619
2024  non-null: 84  min: 0.0222  max: 0.5658


In [27]:
# Spot check: manually verify one city-year end-to-end.
check_year = target_years[0]
check_city = ht_df[HT_INCOME_CONFIG["city_key"]].dropna().iloc[0]
row = ht_df.loc[ht_df[HT_INCOME_CONFIG["city_key"]] == check_city].iloc[0]

print(f"City          : {check_city}")
print(f"Year          : {check_year}")
print(f"HCOST (mo)    : {row[f'HCOST{check_year}']}")
print(f"TCOST (mo)    : {row[f'TCOST{check_year}']}")
print(f"HTCOST (mo)   : {row[f'HTCOST{check_year}']}  (= HCOST + TCOST)")
print(f"Median income : {row[f'MEDINC{check_year}']}")
print(f"Income share  : {row[f'HPLUST{check_year}']:.4f}  (= HTCOST x 12 / income, expect 0-1)")


City          : Alpine
Year          : 2019
HCOST (mo)    : 1829.0
TCOST (mo)    : 762.0
HTCOST (mo)   : 2591.0  (= HCOST + TCOST)
Median income : 135124.0
Income share  : 0.2301  (= HTCOST x 12 / income, expect 0-1)


In [28]:
# Build the export DataFrame: city geometry + H+T data.
export_df = city_area_shp[[HT_INCOME_CONFIG["city_key"], "SUBAREA", "CO_NAME", "SHAPE"]].merge(
    ht_df, on=HT_INCOME_CONFIG["city_key"], how="left"
)

export_df["SUBAREA"] = export_df["SUBAREA"].replace(HT_INCOME_CONFIG["workshop_area_lookup"])

print("Pre-collapse rows:", len(export_df))
print("Pre-collapse cols:", len(export_df.columns))


Pre-collapse rows: 109
Pre-collapse cols: 46


In [29]:
# Drop intermediate derived columns that are not part of the final schema.
# HPLUST{YYYY} is already the income share — written directly in the loop above.
drop_prefixes = ("bg_count_",)
drop_cols = [col for col in export_df.columns if col.startswith(drop_prefixes)]
if drop_cols:
    export_df = export_df.drop(columns=drop_cols)

# Fill NaN in cost columns with 0 (no data = 0, not unknown).
cost_cols = []
for year in target_years:
    cost_cols.extend([f"HCOST{year}", f"TCOST{year}", f"HTCOST{year}", f"HPLUST{year}"])
existing_cost_cols = [col for col in cost_cols if col in export_df.columns]
export_df[existing_cost_cols] = export_df[existing_cost_cols].fillna(0)

# Reorder: HCOST, TCOST, HTCOST (monthly), HPLUST (share), MEDINC, TOTPOP.
ordered_cols = [HT_INCOME_CONFIG["city_key"], "SUBAREA", "CO_NAME", "SHAPE"]
for prefix in ["HCOST", "TCOST", "HTCOST", "HPLUST", "MEDINC", "TOTPOP"]:
    for year in target_years:
        col = f"{prefix}{year}"
        if col in export_df.columns:
            ordered_cols.append(col)
remaining_cols = [col for col in export_df.columns if col not in ordered_cols]
export_df = export_df[ordered_cols + remaining_cols]

print("Final export columns:")
print(export_df.columns.tolist())
export_df.head()


Final export columns:
['CITYAREA', 'SUBAREA', 'CO_NAME', 'SHAPE', 'HCOST2019', 'HCOST2020', 'HCOST2021', 'HCOST2022', 'HCOST2023', 'HCOST2024', 'TCOST2019', 'TCOST2020', 'TCOST2021', 'TCOST2022', 'TCOST2023', 'TCOST2024', 'HTCOST2019', 'HTCOST2020', 'HTCOST2021', 'HTCOST2022', 'HTCOST2023', 'HTCOST2024', 'HPLUST2019', 'HPLUST2020', 'HPLUST2021', 'HPLUST2022', 'HPLUST2023', 'HPLUST2024', 'MEDINC2019', 'MEDINC2020', 'MEDINC2021', 'MEDINC2022', 'MEDINC2023', 'MEDINC2024', 'TOTPOP2019', 'TOTPOP2020', 'TOTPOP2021', 'TOTPOP2022', 'TOTPOP2023', 'TOTPOP2024']


,CITYAREA,SUBAREA,CO_NAME,SHAPE,HCOST2019,HCOST2020,HCOST2021,HCOST2022,HCOST2023,HCOST2024,TCOST2019,TCOST2020,TCOST2021,TCOST2022,TCOST2023,TCOST2024,HTCOST2019,HTCOST2020,HTCOST2021,HTCOST2022,HTCOST2023,HTCOST2024,HPLUST2019,HPLUST2020,HPLUST2021,HPLUST2022,HPLUST2023,HPLUST2024,MEDINC2019,MEDINC2020,MEDINC2021,MEDINC2022,MEDINC2023,MEDINC2024,TOTPOP2019,TOTPOP2020,TOTPOP2021,TOTPOP2022,TOTPOP2023,TOTPOP2024
0,Alpine,Utah County North,UTAH,"{""rings"": [[[433081.68319999985, 4477344.8058]...",1829.0,1858.0,2074.0,2640.0,2867.0,4341.0,762.0,608.0,701.0,716.0,738.0,749.0,2591.0,2466.0,2775.0,3356.0,3605.0,5090.0,0.230100,0.206335,0.226140,0.239112,0.270432,0.376088,135124.0,143417.0,147254.0,168423.0,159966.0,162409.0,9794.0,10208.0,9756.0,9484.0,9649.0,9566.0
1,Alta,NA,SALT LAKE,"{""rings"": [[[449359.0999999996, 4492074], [449...",0.0,0.0,0.0,0.0,0.0,0.0,504.0,336.0,408.0,447.0,474.0,441.0,504.0,336.0,408.0,447.0,474.0,441.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,American Fork,Utah County North,UTAH,"{""rings"": [[[433815.22979999986, 4466632.21309...",1521.0,1600.0,1788.0,2213.0,2400.0,3366.0,429.0,352.0,394.0,406.0,418.0,426.0,1950.0,1952.0,2182.0,2619.0,2818.0,3792.0,0.277837,0.270329,0.287149,0.312331,0.313468,0.408023,84222.0,86650.0,91186.0,100624.0,107877.0,111523.0,33624.0,33780.0,34725.0,36227.0,37988.0,39785.0
3,Balance of BOX ELDER,NA,BOX ELDER,"{""rings"": [[[417479, 4578625.4], [417425.5, 45...",0.0,0.0,0.0,0.0,0.0,0.0,1235.0,914.0,1014.0,1140.0,1340.0,1141.0,1235.0,914.0,1014.0,1140.0,1340.0,1141.0,0.185410,0.128183,0.132021,0.131769,0.148530,0.117833,79931.0,85565.0,92167.0,103818.0,108261.0,116198.0,5295.0,5126.0,5328.0,5582.0,6109.0,6703.0
4,Benjamin,Utah County South,UTAH,"{""rings"": [[[438601.23319999967, 4437471.4559]...",0.0,0.0,0.0,0.0,0.0,0.0,587.0,486.0,550.0,551.0,558.0,536.0,587.0,486.0,550.0,551.0,558.0,536.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
export_fc = os.path.join(gdb2, "Affordability_Housing_Transportation_Costs")

if arcpy.Exists(export_fc):
    arcpy.management.Delete(export_fc)

export_df.spatial.to_featureclass(location=export_fc, sanitize_columns=False)
print("Exported:", export_fc)


Exported: .\Outputs\Affordability_Housing_Transportation_Costs.gdb\Affordability_Housing_Transportation_Costs


## 9. Validate Export

In [31]:
export_check = pd.DataFrame.spatial.from_featureclass(export_fc)

print("Export shape:", export_check.shape)
print(export_check.columns.tolist())

# Quick look at the first year's key output columns.
check_cols = [
    "CITYAREA",
    "SUBAREA",
    "CO_NAME",
    f"HCOST{target_years[0]}",
    f"TCOST{target_years[0]}",
    f"HTCOST{target_years[0]}",  # monthly combined cost (dollars)
    f"HPLUST{target_years[0]}",  # 0-1 income share
]
print(export_check[check_cols].head())


Export shape: (109, 41)
['OBJECTID', 'CITYAREA', 'SUBAREA', 'CO_NAME', 'HCOST2019', 'HCOST2020', 'HCOST2021', 'HCOST2022', 'HCOST2023', 'HCOST2024', 'TCOST2019', 'TCOST2020', 'TCOST2021', 'TCOST2022', 'TCOST2023', 'TCOST2024', 'HTCOST2019', 'HTCOST2020', 'HTCOST2021', 'HTCOST2022', 'HTCOST2023', 'HTCOST2024', 'HPLUST2019', 'HPLUST2020', 'HPLUST2021', 'HPLUST2022', 'HPLUST2023', 'HPLUST2024', 'MEDINC2019', 'MEDINC2020', 'MEDINC2021', 'MEDINC2022', 'MEDINC2023', 'MEDINC2024', 'TOTPOP2019', 'TOTPOP2020', 'TOTPOP2021', 'TOTPOP2022', 'TOTPOP2023', 'TOTPOP2024', 'SHAPE']
               CITYAREA            SUBAREA    CO_NAME  HCOST2019  TCOST2019  \
0                Alpine  Utah County North       UTAH       1829        762   
1                  Alta                 NA  SALT LAKE          0        504   
2         American Fork  Utah County North       UTAH       1521        429   
3  Balance of BOX ELDER                 NA  BOX ELDER          0       1235   
4              Benjamin  Utah Cou

In [32]:
# Per-year range check on the income share (stored in HPLUST columns).
# Expected: values between 0.0 and ~0.6; max above 1.0 would indicate a calculation error.
print("Export feature class:", export_fc)
print("Rows   :", len(export_check))
print("Columns:", len(export_check.columns))
print()
for year in target_years:
    s = export_check[f"HPLUST{year}"]
    print(f"{year}  HPLUST (share) min: {s.min():.4f}  max: {s.max():.4f}")


Export feature class: .\Outputs\Affordability_Housing_Transportation_Costs.gdb\Affordability_Housing_Transportation_Costs
Rows   : 109
Columns: 41

2019  HPLUST (share) min: 0.0000  max: 0.4289
2020  HPLUST (share) min: 0.0000  max: 0.4239
2021  HPLUST (share) min: 0.0000  max: 0.4206
2022  HPLUST (share) min: 0.0000  max: 0.4198
2023  HPLUST (share) min: 0.0000  max: 0.4619
2024  HPLUST (share) min: 0.0000  max: 0.5658


In [33]:
# Spot check: Alpine in the most recent year.
check_year = target_years[-1]
check_city = "Alpine"

export_check.loc[
    export_check["CITYAREA"] == check_city,
    [
        "CITYAREA",
        f"HCOST{check_year}",
        f"TCOST{check_year}",
        f"HTCOST{check_year}",
        f"HPLUST{check_year}",
    ],
]


,CITYAREA,HCOST2024,TCOST2024,HTCOST2024,HPLUST2024
0,Alpine,4341,749,5090,0.376088
